# 香港特区`CBERS-04`卫星`MUX`影像元数据读取和`sqlite`数据库导出

## 程序包初始化

### 内建程序包

In [1]:
import io, sys, os; nb_dir = os.getcwd(); 
sys.path.append(nb_dir); 

In [2]:
import re; 
import collections as coll, itertools as it; 
from __future__ import print_function; 

In [3]:
import sqlite3;

### 第三方和自定义程序包

In [4]:
import numpy as np, pandas as pd; 

In [5]:
py_pkg_dir = os.path.normpath(
    os.path.join(
        nb_dir, os.pardir, os.pardir, 
        "Python_Package"
    )
); 
sys.path.append(py_pkg_dir); 

In [6]:
import cresda_metadata_parse as cresda; 
cresda = cresda.importlib.reload(cresda); 

## 影像元数据读取和格式转换

In [7]:
rs_meta_dir = os.path.normpath(
    os.path.join(
        nb_dir, os.pardir, os.pardir, 
        os.pardir, "Source", "Imagery"
    )
); 

### 元数据读取和格式转换工具初始化

In [8]:
mux_finder = cresda.cb04.MUX(); 
mux_finder.source_dir = rs_meta_dir; 
mux_finder.target_dir = rs_meta_dir; 
mux_meta = cresda.metadata.CresdaProductMetadata(); 
mux_meta_format = cresda.metadata.CresdaMetadataFormatter(); 

### 影像存放路径定位

### 文件名匹配

In [9]:
cb04_mux_scenes = tuple(scene for scene in mux_finder.traverse()); 

### 元数据`xml`文件的打开与读取

In [10]:
recs = list(); 
for scene in cb04_mux_scenes: 
    path_xml = scene.target[0]; 
    rec = mux_meta.from_xml(path_xml); 
    rec.filename = os.path.basename(scene.source); 
    recs.append(rec.__dict__); 
recs = pd.DataFrame(recs); 
recs.set_index(
    "filename", inplace=True, 
    verify_integrity=True
); 

### 数据入库前的格式转换
* `xml`文件中的小数 (经纬度, 投影XY坐标) 字符串, 转换为`IEEE 754`标准64位浮点数; 
* `xml`文件中的时间, 精度为整秒或百分之一秒, 一律转换为`UT 1970-01-01T00:00:00Z`以来的毫秒数, 即`Unix`时间戳的一千倍, 以64位有符号整型存储. 

In [11]:
def pd_obj_to_fp64(series): 
    if all(
        rec is None \
        or re.match(mux_meta_format.decimal_filter, rec) 
        for rec in series
    ): 
        return series.astype(np.float64); 

def pd_obj_to_timestamp(series): 
    if all(
        rec is None \
        or re.match(mux_meta_format.date_time_filter, rec) \
        or re.match(mux_meta_format.precise_date_time_filter, rec)
        for rec in series
    ): 
        return pd.Series(
            (np.datetime64(rec, "ms").astype(np.int64) for rec in series), 
            index=series.index, dtype=np.int64
        ); 

In [12]:
for field in recs: 
    if recs[field].dtype != np.object: 
        continue; 
    if re.match(r".*_(?:lat|long|x|y)\Z", field): 
        conv = pd_obj_to_fp64(recs[field]); 
        if conv is not None: 
            recs[field] = conv; 
    elif re.match(r"\Asun_.*", field): 
        conv = pd_obj_to_fp64(recs[field]); 
        if conv is not None: 
            recs[field] = conv; 
    elif re.match(r".*_(?:date|time)\Z", field): 
        conv = pd_obj_to_timestamp(recs[field]); 
        if conv is not None: 
            recs[field] = conv; 

## `xml`元数据读取结果入库

In [13]:
rs_meta_items_sqlite = sqlite3.connect(os.path.join(
    nb_dir, "CBERS-04_metadata.sqlite"
) ); 

In [14]:
recs.to_sql(
    con=rs_meta_items_sqlite, flavor="sqlite", 
    name="info_metadata_cb04", if_exists="replace", 
    index=True
); 

In [15]:
rs_img_stat = pd.read_sql(
    """	
    SELECT 
        filename, scene_date, 
        scene_center_lat As Lat, 
        scene_center_long As Lon
	From info_metadata_cb04
    """, con=rs_meta_items_sqlite
); 
rs_img_stat.set_index("filename", inplace=True); 
rs_img_stat

,scene_date,Lat,Lon
filename,,,
CB04-MUX-371-75-20151021-L20002658001.TIF,1445397142480,22.399896,113.899098
CB04-MUX-371-75-20160228-L20002779732.TIF,1456629157700,22.399671,113.914153
CB04-MUX-371-75-20171209-L20003295081.TIF,1512788572600,22.399723,113.807733
CB04-MUX-371-75-20180323-L20003377915.TIF,1521773950990,22.398903,113.848824
CB04-MUX-371-75-20180921-L20003494311.TIF,1537498298750,22.399825,113.867681
CB04-MUX-371-75-20241012-L20005228726.TIF,1728700729020,22.399650,113.843114
CB04-MUX-371-75-20241107-L20005258293.TIF,1730947037480,22.399800,113.872103
CB04-MUX-371-75-20241229-L20005301017.TIF,1735439665570,22.399680,113.855350


In [16]:
rs_meta_items_sqlite.execute("VACUUM"); 
rs_meta_items_sqlite.commit(); 
rs_meta_items_sqlite.close(); 